# Week 2 – Agent with tools and memory (smolagents)
**COS30018 Intelligent Systems · Anas Al Azad**

Last week I wrote the "send prompt → get answer" code myself. This week the **smolagents** framework does the loop for me: the model writes Python code, smolagents runs it, shows the model the result, and repeats until the model calls `final_answer()` – the **Thought → Code → Observation** cycle.

* cloud model: `meta-llama/Llama-3.3-70B-Instruct` through Hugging Face (free token)
* local model: `Qwen/Qwen2.5-1.5B-Instruct` with `TransformersModel` (downloads ~3 GB the first time)
* tool: `DuckDuckGoSearchTool()` (web search)

**Before running:** `HF_TOKEN=hf_...` is in the `.env` file in the top folder of the repo, and `pip install "smolagents[toolkit]" "smolagents[transformers]"` is done. Then *Kernel → Restart & Run All* (the local model part can take 10+ minutes).

In [ ]:
import dataclasses
import json
import os
import platform
import time
from pathlib import Path

import pandas as pd
import smolagents
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()   # reads HF_TOKEN from the .env file (this folder or the folders above)
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "No HF_TOKEN found - put HF_TOKEN=hf_... in the .env file in the top folder of the repo"

CLOUD_MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"
LOCAL_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_LOCAL = True     # False = skip the local model (it downloads ~3 GB and is slow on a laptop)
MAX_STEPS = 6        # (my addition) stop a confused model after 6 cycles instead of the default 20

TASK_SUM = "Calculate the sum of numbers from 1 to 50"
TASK_SEARCH = "Search for the current population of Melbourne, then calculate 15% of that number."
TASK_WEATHER = "What is the current weather in Melbourne?"

Path("results").mkdir(exist_ok=True)
results = {"run_at": time.strftime("%Y-%m-%d %H:%M"), "python": platform.python_version(),
           "smolagents": smolagents.__version__, "cloud_model": CLOUD_MODEL_ID, "local_model": LOCAL_MODEL_ID,
           "max_steps": MAX_STEPS}
print("smolagents", smolagents.__version__, "| Python", platform.python_version(), "| HF token found:", bool(HF_TOKEN))

**Helper** – after each run I turn `agent.memory.steps` into a small table (one row per Thought → Code → Observation cycle), so I can count the cycles and see the time and tokens of each one.

In [ ]:
from smolagents.memory import ActionStep


def describe_run(agent, answer, seconds, show=True):
    rows = []
    for step in agent.memory.steps:
        if not isinstance(step, ActionStep):
            continue
        tu = step.token_usage
        rows.append({"step": step.step_number,
                     "seconds": round(step.timing.duration or 0, 2),
                     "tokens in": tu.input_tokens if tu else None,
                     "tokens out": tu.output_tokens if tu else None,
                     "searched?": "web_search(" in (step.code_action or ""),
                     "code (first line)": (step.code_action or "").strip().split("\n")[0][:70],
                     "observation": (step.observations or "").strip().replace("\n", " ")[:90],
                     "error": str(step.error)[:90] if step.error else "",
                     "final answer?": bool(step.is_final_answer)})
    if show:
        display(pd.DataFrame(rows).set_index("step"))
    return {"answer": str(answer), "seconds": round(seconds, 1), "n_steps": len(rows), "steps": rows,
            "tokens_in": sum(r["tokens in"] or 0 for r in rows), "tokens_out": sum(r["tokens out"] or 0 for r in rows),
            "searches": sum(r["searched?"] for r in rows), "errors": sum(bool(r["error"]) for r in rows),
            "finished": any(r["final answer?"] for r in rows)}

## Step 1 – install smolagents + get a Hugging Face token
Terminal: `pip install "smolagents[toolkit]" "smolagents[transformers]"`. Token: huggingface.co → profile picture → *Settings* → *Access Tokens* → *Create new token* (type **Read** is enough, or a fine-grained one with *"Make calls to Inference Providers"* ticked). It goes in `.env`, never in the notebook.

## Step 2 – a basic cloud agent with no tools
Same code as the sheet, except the token comes from `.env` (the sheet types it into the code).

In [ ]:
from smolagents import CodeAgent, InferenceClientModel

model = InferenceClientModel(
    model_id=CLOUD_MODEL_ID,
    token=HF_TOKEN
)

agent = CodeAgent(tools=[], model=model, max_steps=MAX_STEPS)

start = time.time()
result = agent.run(TASK_SUM)
took = time.time() - start
print(result)

results["no_tools"] = describe_run(agent, result, took)
results["no_tools"]["correct"] = str(result).strip() in ("1275", "1275.0")
print(f"{results['no_tools']['n_steps']} cycle(s), {took:.1f} s, right answer (1275)? {results['no_tools']['correct']}")

## Step 3 – add web search, and a task that needs two steps
The agent has to (1) search the web and (2) do maths with what it found, so the loop runs at least twice. In the log below each **Step** box is one cycle: the model's **Thought** + the **code** it wrote ("Executing parsed code"), then the **Observation** ("Execution logs" – what the code printed), which goes back to the model for the next step.

The sheet warns DuckDuckGo can rate-limit a lab full of students, so first I check the search tool works and switch to smolagents' `WebSearchTool` (Bing) if it doesn't. Both are called `web_search`, so the agent code doesn't change.

In [ ]:
from smolagents import DuckDuckGoSearchTool, WebSearchTool

search_tool, search_note = DuckDuckGoSearchTool(), None
try:
    test = search_tool("Melbourne population")
    print("DuckDuckGo works:", test[:200].replace("\n", " "), "...")
except Exception as e:
    search_note = f"DuckDuckGo failed ({type(e).__name__}: {str(e)[:100]}) -> switched to WebSearchTool(engine='bing')"
    print(search_note)
    search_tool = WebSearchTool(engine="bing")
results["search_tool"] = type(search_tool).__name__
results["search_note"] = search_note

In [ ]:
agent = CodeAgent(
    tools=[search_tool],
    model=model,
    max_steps=MAX_STEPS,
)

start = time.time()
result = agent.run(TASK_SEARCH)
took = time.time() - start
print(result)

results["search"] = describe_run(agent, result, took)
try:
    population = float(result) / 0.15        # the number the agent must have used
    results["search"]["population_used"] = round(population)
    results["search"]["looks_right"] = 4_500_000 <= population <= 5_700_000   # Melbourne is ~5 million people
except (TypeError, ValueError):
    results["search"]["looks_right"] = False
pop = results["search"].get("population_used")
print(f"{results['search']['n_steps']} cycles, {took:.1f} s, {results['search']['searches']} web search(es) | "
      "population used: " + (f"{pop:,}" if pop else "couldn't read a number from the answer"))

## Step 4 – what the framework remembered by itself (`agent.memory.steps`)
The sheet's loop. Each step is huge when printed (it also stores every message that was sent to the model, including the long system prompt), so I cut each one to 400 characters.

In [ ]:
for step in agent.memory.steps:
    print(str(step)[:400])
    print("---")

sizes = [len(str(step)) for step in agent.memory.steps]
print(f"\n{len(agent.memory.steps)} memory entries, {sum(sizes):,} characters in total")
print("\nwhat smolagents records for every action step:", [f.name for f in dataclasses.fields(ActionStep)])
print("the system prompt it wrote for me is", len(agent.memory.system_prompt.system_prompt), "characters long")
results["memory"] = {"entries": len(agent.memory.steps), "types": [type(s).__name__ for s in agent.memory.steps],
                     "characters": sum(sizes), "fields": [f.name for f in dataclasses.fields(ActionStep)],
                     "system_prompt_chars": len(agent.memory.system_prompt.system_prompt)}

**Extra – does the memory actually work?** `reset=False` keeps the old steps, so I can ask a follow-up that only makes sense if it remembers the first task.

In [ ]:
FOLLOW_UP = "Using the population number you found before (don't search again), what is 30% of it?"
before = len(agent.memory.steps)
start = time.time()
result2 = agent.run(FOLLOW_UP, reset=False)
took = time.time() - start
print(result2)

new_steps = [s for s in agent.memory.steps[before:] if isinstance(s, ActionStep)]
searched_again = any("web_search(" in (s.code_action or "") for s in new_steps)
results["memory"]["follow_up"] = {"question": FOLLOW_UP, "answer": str(result2), "seconds": round(took, 1),
                                  "n_steps": len(new_steps), "searched_again": searched_again}
try:
    expected = results["search"]["population_used"] * 0.30
    results["memory"]["follow_up"]["matches"] = abs(float(result2) - expected) < 0.01 * expected
except (KeyError, TypeError, ValueError):
    results["memory"]["follow_up"]["matches"] = None
print(f"{len(new_steps)} new cycle(s), searched again: {searched_again}, "
      f"= 30% of the same population: {results['memory']['follow_up']['matches']}")

## Step 5 – same agent, local model (`TransformersModel`)
Only the model line changes. The first run downloads the model (~3 GB) into the Hugging Face cache, then it runs on my laptop's CPU. The sheet warns it may be slow or fail – so it's in a `try`, and the error is saved if it breaks. (`max_new_tokens=1024` is part of the model line, so a rambling answer can't take forever.)

In [ ]:
from smolagents import TransformersModel

local_model = None
if RUN_LOCAL:
    try:
        start = time.time()
        local_model = TransformersModel(model_id=LOCAL_MODEL_ID, max_new_tokens=1024)
        load_s = time.time() - start
        print(f"loaded {LOCAL_MODEL_ID} in {load_s:.0f} s")

        agent = CodeAgent(
            tools=[search_tool],
            model=local_model,
            max_steps=MAX_STEPS,
        )
        start = time.time()
        result = agent.run(TASK_SEARCH)
        took = time.time() - start
        print(result)
        results["local"] = describe_run(agent, result, took)
        results["local"]["load_s"] = round(load_s)
        try:
            results["local"]["population_used"] = round(float(result) / 0.15)
            results["local"]["looks_right"] = 4_500_000 <= float(result) / 0.15 <= 5_700_000
        except (TypeError, ValueError):
            results["local"]["looks_right"] = False
    except Exception as e:
        print("Local model failed:", type(e).__name__, str(e)[:300])
        results["local"] = {"failed": f"{type(e).__name__}: {str(e)[:300]}"}

## Step 6 – compare: same task, cloud vs local
The sheet's code: two agents that only differ in the model line, quiet logs (`LogLevel.ERROR`), and a `run_and_time` helper. (my change: I reuse the local model loaded above instead of loading another 3 GB copy.)

In [ ]:
from smolagents import LogLevel

# Setup cloud vs local agents
# Use verbosity_level=LogLevel.ERROR to surpress log

cloud_model = InferenceClientModel(
    model_id=CLOUD_MODEL_ID,
    token=HF_TOKEN
)
cloud_agent = CodeAgent(
    tools=[search_tool],
    model=cloud_model,
    verbosity_level=LogLevel.ERROR,
    max_steps=MAX_STEPS,
)

local_agent = None
if local_model is not None:
    local_agent = CodeAgent(
        tools=[search_tool],
        model=local_model,
        verbosity_level=LogLevel.ERROR,
        max_steps=MAX_STEPS,
    )

In [ ]:
def run_and_time(agent, task):
    start = time.time()
    result = agent.run(task)
    elapsed = time.time() - start
    return result, elapsed


task = TASK_WEATHER

cloud_result, cloud_time = run_and_time(cloud_agent, task)
print(f"Cloud: {cloud_time:.2f}s\n{cloud_result}\n")
results["compare"] = {"task": task, "cloud": describe_run(cloud_agent, cloud_result, cloud_time, show=False)}

if local_agent is not None:
    try:
        local_result, local_time = run_and_time(local_agent, task)
        print(f"Local: {local_time:.2f}s\n{local_result}")
        results["compare"]["local"] = describe_run(local_agent, local_result, local_time, show=False)
    except Exception as e:
        print("Local failed:", type(e).__name__, str(e)[:300])
        results["compare"]["local"] = {"failed": f"{type(e).__name__}: {str(e)[:300]}"}

In [ ]:
def col(r):
    if not r:
        return {"finished?": "skipped"}
    if "failed" in r:
        return {"finished?": "crashed: " + r["failed"][:60]}
    return {"finished?": "yes" if r["finished"] else f"no (hit max_steps={MAX_STEPS})",
            "cycles": r["n_steps"], "seconds": r["seconds"], "web searches": r["searches"],
            "errors fixed along the way": r["errors"], "tokens in / out": f"{r['tokens_in']:,} / {r['tokens_out']:,}",
            "final answer": r["answer"][:80]}


comparison = pd.DataFrame({"Melbourne 15% – cloud": col(results.get("search")),
                           "Melbourne 15% – local": col(results.get("local")),
                           "weather – cloud": col(results["compare"].get("cloud")),
                           "weather – local": col(results["compare"].get("local"))})
comparison

## Step 7 – the agent in a chat window (`GradioUI`)
`GradioUI(agent).launch()` would block the notebook (it runs in debug mode) and makes a public share link by default, so I build the app and launch it myself – local only. The chat keeps the agent's memory between messages (`reset_agent_memory=False`), so I can ask follow-ups. Opens at **http://127.0.0.1:7860** (or 7861 if that port is taken – see the link below).

In [ ]:
from smolagents import GradioUI

chat_agent = CodeAgent(tools=[search_tool], model=cloud_model, max_steps=MAX_STEPS)
ui = GradioUI(chat_agent)                 # reset_agent_memory=False -> remembers earlier messages
app = ui.create_app()
app.launch(share=False, prevent_thread_lock=True)
results["gradio"] = True

## Checkpoint questions
Answers are built from the numbers of this run (so they always match what happened).

In [ ]:
s, nt = results["search"], results["no_tools"]
loc, cmp_ = results.get("local") or {}, results["compare"]
fu = results["memory"].get("follow_up", {})
final_step = next((r["step"] for r in s["steps"] if r["final answer?"]), None)

a1 = ("In Week 1 I wrote every request myself: build the prompt, call the API, read the reply, keep the chat history in "
      "a list and time things by hand. smolagents did all of that plus: it wrote a "
      f"{results['memory']['system_prompt_chars']:,}-character system prompt that explains the tools and the Thought → "
      "Code → Observation format, pulled the code out of the model's reply, ran it, sent back what it printed (or the "
      "error so the model could fix it), looped until `final_answer()` was called, and logged every step with its "
      "time and tokens in `agent.memory.steps`" + (f" – and with `reset=False` it remembered the first task, so the "
                                                  f"follow-up took {fu.get('n_steps')} cycle(s) and "
                                                  f"{'did' if fu.get('searched_again') else 'did not'} search again." if fu else "."))
if s["finished"]:
    a2 = (f"{s['n_steps']} cycle(s) for the Melbourne task ({s['searches']} web search(es)). It stopped at step "
          f"{final_step} because the code it wrote called `final_answer(...)` – that's the signal smolagents waits for. "
          f"(The no-tools sum task took {nt['n_steps']} cycle(s).)")
else:
    a2 = (f"It used all {s['n_steps']} cycles without calling `final_answer()`, so it was stopped by "
          f"max_steps={MAX_STEPS} – smolagents then asks the model for a best-guess answer.")


def short(r):
    if not r:
        return "skipped"
    if "failed" in r:
        return "crashed (" + r["failed"][:80] + ")"
    return f"{r['n_steps']} cycles, {r['seconds']} s, answer: {r['answer'][:60]}"


a3 = ("Stayed the same: the agent code, the search tool, the prompt and the memory – only the `model=` line changed. "
      f"Changed: speed and quality. Melbourne task – cloud: {short(s)}; local: {short(loc)}. Weather – cloud: "
      f"{short(cmp_.get('cloud'))}; local: {short(cmp_.get('local'))}. The local model also had to be downloaded and "
      "loaded first" + (f" ({loc.get('load_s')} s)" if loc.get("load_s") is not None else "") + ".")
if not loc or "failed" in loc:
    a4 = "No – " + ("I skipped it." if not loc else f"it crashed: {loc['failed'][:150]}")
elif loc.get("looks_right"):
    a4 = (f"Yes – it found a population of about {loc['population_used']:,} and gave {loc['answer']}, in "
          f"{loc['n_steps']} cycles" + (f" (it hit {loc['errors']} error(s) on the way and fixed them)." if loc["errors"] else "."))
elif loc.get("finished"):
    a4 = (f"Not really – it finished, but the answer ({loc['answer'][:80]}) isn't 15% of Melbourne's ~5 million people, "
          "so it used a wrong number or did the maths wrong.")
else:
    a4 = (f"No – it ran out of steps (max_steps={MAX_STEPS}) after {loc['errors']} error(s); the small model kept "
          "writing code that didn't run or didn't follow the format.")

answers = {"q1": a1, "q2": a2, "q3": a3, "q4": a4}
results["checkpoint"] = answers
for q, a in zip(["What did the framework do for you that you would have written manually in Week 1?",
                 "How many Thought → Code → Observation cycles ran before the agent gave a final answer, and what triggered it to stop?",
                 "What changed when you swapped from cloud to local, and what stayed the same?",
                 "Did your local model complete the task successfully? If not, what did it get wrong?"], answers.values()):
    display(Markdown(f"**{q}**\n\n{a}"))

with open("results/lab2_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("saved results/lab2_results.json (my worklog builder reads this)")